In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from notebookutils import mssparkutils


In [ ]:
print("SourceName =", SourceName)
print("IncrementalColumn =", IncrementalColumn)
print("BusinessKey =", BusinessKey)
print("WatermarkDate =", WatermarkDate)
print("LoadType =", LoadType)

In [ ]:
silver_table = f"Silver_{SourceName}"

if SourceName == "Patients":
    gold_table = "Gold_Dim_Patient"

elif SourceName == "Doctors":
    gold_table = "Gold_Dim_Doctor"

elif SourceName == "Hospitals":
    gold_table = "Gold_Dim_Hospital"

elif SourceName == "Claims":
    gold_table = "Gold_Fact_Claims"

elif SourceName == "Pharmacy":
    gold_table = "Gold_Fact_Pharmacy"

print("Silver Table =", silver_table)
print("Gold Table =", gold_table)

In [ ]:
silver_df = spark.table(silver_table)

print(
    "Silver Count =",
    silver_df.count()
)

In [ ]:
if BusinessKey not in silver_df.columns:

    raise Exception(
        f"{BusinessKey} not found in Silver Table"
    )

if IncrementalColumn not in silver_df.columns:

    raise Exception(
        f"{IncrementalColumn} not found in Silver Table"
    )

print("Schema Validation Passed")

In [ ]:
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [ ]:
if LoadType == "Full":

    process_df = silver_df

else:

    process_df = silver_df.filter(
        col(IncrementalColumn) > WatermarkDate
    )

In [ ]:
record_count = process_df.count()

print(
    "Records To Process =",
    record_count
)

if record_count == 0:

    print("No New Records Found")

    mssparkutils.notebook.exit("NO_DATA")

In [ ]:
process_df = process_df.filter(
    col(BusinessKey).isNotNull()
)


In [ ]:
process_df = process_df.dropDuplicates(
    [BusinessKey]
)

In [ ]:
process_df = process_df.na.drop(how="all")

In [ ]:
if SourceName == "Patients":

    process_df = process_df.withColumn(
        "PatientID",
        col("PatientID").cast("string")
    )

elif SourceName == "Doctors":

    process_df = process_df.withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    ).withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Hospitals":

    process_df = process_df.withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Claims":

    process_df = process_df.withColumn(
        "ClaimID",
        col("ClaimID").cast("string")
    ).withColumn(
        "PatientID",
        col("PatientID").cast("string")
    ).withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    ).withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Pharmacy":

    process_df = process_df.withColumn(
        "PatientID",
        col("PatientID").cast("string")
    ).withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    )

In [ ]:
from pyspark.sql.functions import *

In [ ]:
for c in process_df.columns:

    if c.upper().endswith("DATE"):

        if dict(process_df.dtypes)[c] in ["bigint","long","int"]:

            process_df = process_df.withColumn(
                c,
                to_timestamp(
                    from_unixtime(col(c)/1000)
                )
            )

In [ ]:
print(process_df.printSchema())

In [ ]:
if not spark.catalog.tableExists(gold_table):

    process_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option(
            "mergeSchema",
            "true"
        ) \
        .saveAsTable(
            gold_table
        )

    print("Initial Gold Table Created")
    print("Records Processed =", record_count)

In [ ]:
merge_condition = (
    f"t.{BusinessKey}"
    f" = "
    f"s.{BusinessKey}"
)

print(
    "Merge Condition =",
    merge_condition
)

In [ ]:
gold_delta = DeltaTable.forName(
    spark,
    gold_table
)

(
    gold_delta.alias("t")
    .merge(
        process_df.alias("s"),
        merge_condition
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
print(
    "Gold Merge Completed"
)

In [ ]:
if SourceName == "Patients":

    process_df = process_df.withColumn(
        "PatientID",
        col("PatientID").cast("string")
    )

elif SourceName == "Doctors":

    process_df = process_df.withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    ).withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Hospitals":

    process_df = process_df.withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Claims":

    process_df = process_df.withColumn(
        "ClaimID",
        col("ClaimID").cast("string")
    ).withColumn(
        "PatientID",
        col("PatientID").cast("string")
    ).withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    ).withColumn(
        "HospitalID",
        col("HospitalID").cast("string")
    )

elif SourceName == "Pharmacy":

    process_df = process_df.withColumn(
        "PatientID",
        col("PatientID").cast("string")
    ).withColumn(
        "DoctorID",
        col("DoctorID").cast("string")
    )

In [ ]:

spark.table(gold_table).printSchema()


In [ ]:
mssparkutils.notebook.exit(
    str(record_count)
)